Model Training

Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,OrdinalEncoder,StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error,mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV


In [2]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.3 MB/s eta 0:00:00


In [3]:
from sklearn.linear_model import LinearRegression,Lasso,Ridge,ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor,VotingRegressor,BaggingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [4]:
df=pd.read_csv('/content/MODEL_DATA.csv')
df.head()

,brand,model,color,year,power_ps,transmission_type,fuel_type,fuel_consumption_l_100km,fuel_consumption_g_km,mileage_in_km,price_in_euro
0,ford,Kuga,black,2023,190.0,Automatic,Hybrid,5.4,124.0,100.0,38490.0
1,hyundai,i10,black,2018,67.0,Manual,Petrol,4.6,106.0,27782.0,11555.0
2,audi,Q4 e-tron,grey,2021,170.0,Automatic,Electric,5.8,0.0,4247.0,48886.0
3,honda,CR-V,red,2018,155.0,Automatic,Petrol,7.5,175.0,57000.0,24490.0
4,kia,Sportage,black,2023,150.0,Manual,Petrol,5.9,150.0,7500.0,34990.0


In [5]:
x=df.drop('price_in_euro',axis=1)
y=df['price_in_euro']

Split data into Dependent and Independent features

Split the dataset into training and testing sets.

In [6]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.30,random_state=30)

Encoding

In [7]:
categorical_features_lb = ['fuel_type', 'brand', 'model', 'color']
categorical_features_one = ['transmission_type']
numeric_features = ['mileage_in_km', 'fuel_consumption_l_100km', 'fuel_consumption_g_km']

onehot_scaled = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

ordinal_scaled = Pipeline([
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ('scaler', StandardScaler())
])

numeric_scaled = Pipeline([
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot_scaled', onehot_scaled, categorical_features_one),
        ('ordinal_scaled', ordinal_scaled, categorical_features_lb),
        ('numeric_scaled', numeric_scaled, numeric_features)
    ]
)

X_train_encoded = preprocessor.fit_transform(x_train)
X_test_encoded = preprocessor.transform(x_test)

In [8]:
X_train_encoded=pd.DataFrame(X_train_encoded)
X_test_encoded=pd.DataFrame(X_test_encoded)

Model Building

In [9]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [10]:
estimators=[('rf', RandomForestRegressor()), ('xg', XGBRegressor()), ('cat', CatBoostRegressor(verbose=False)), ('lgbm', LGBMRegressor())]

In [11]:
models={

    'K-Neighbors Regressor':KNeighborsRegressor(),
    'SVR':SVR(kernel='linear'),#

    'LinearRegression':LinearRegression(),#
    'Lasso':Lasso(),#
    'Ridge':Ridge(),#
    'ElasticNet':ElasticNet(),#


    'Decision Tree':DecisionTreeRegressor(),

    'Random Forest Regressor':RandomForestRegressor(),
    'VotingRegressor':VotingRegressor(estimators=estimators),
    'BaggingRegressor':BaggingRegressor(estimator=LinearRegression(),bootstrap=True),

    'AdaBoost Regressor':AdaBoostRegressor(),#
    'GradientBoostingRegressor':GradientBoostingRegressor(),
    'XGBRegressor':XGBRegressor(),
    'CatBoosting Regressor':CatBoostRegressor(verbose=False),
    'LGBMRegressor':LGBMRegressor(),

    'StackingRegressor':StackingRegressor(estimators=estimators,final_estimator=XGBRegressor()),
}


In [12]:
trained_model_list = []
model_list = []
r2_list = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train_encoded, y_train)

    # Train evaluation
    y_train_pred = model.predict(X_train_encoded)
    mae_train, rmse_train, r2_train = evaluate_model(y_train, y_train_pred)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print('Model Training Performance')
    print("RMSE:", rmse_train)
    print("MAE:", mae_train)
    print("R2 score:", r2_train * 100)

    # Test evaluation
    y_test_pred = model.predict(X_test_encoded)
    mae_test, rmse_test, r2_test = evaluate_model(y_test, y_test_pred)

    print('\nModel Testing Performance')
    print("RMSE:", rmse_test)
    print("MAE:", mae_test)
    print("R2 score:", r2_test * 100)

    r2_list.append(r2_test)  # save only test R² if you want comparison

    print('=' * 35)
    print('\n')


K-Neighbors Regressor
Model Training Performance
RMSE: 19159.07318118644
MAE: 6345.58326586839
R2 score: 82.48471555697868

Model Testing Performance
RMSE: 26263.504896735463
MAE: 8225.445641101716
R2 score: 67.86012990588874


SVR
Model Training Performance
RMSE: 42164.51030042518
MAE: 13243.408613501893
R2 score: 15.167488719873745

Model Testing Performance
RMSE: 42842.72447705597
MAE: 13488.234790540897
R2 score: 14.475033370237345


LinearRegression
Model Training Performance
RMSE: 38158.42812455698
MAE: 16333.760346749674
R2 score: 30.521704629237878

Model Testing Performance
RMSE: 38918.88186608945
MAE: 16435.234551809746
R2 score: 29.423606692102368




/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.155e+10, tolerance: 9.889e+09
  model = cd_fast.enet_coordinate_descent(


Lasso
Model Training Performance
RMSE: 38158.44656463188
MAE: 16332.52764912193
R2 score: 30.521637478396336

Model Testing Performance
RMSE: 38918.17102612678
MAE: 16433.864990591173
R2 score: 29.426184775553033


Ridge
Model Training Performance
RMSE: 38158.44283644854
MAE: 16333.287142018986
R2 score: 30.52165105484318

Model Testing Performance
RMSE: 38917.70531466647
MAE: 16434.416134926505
R2 score: 29.427873798291316


ElasticNet
Model Training Performance
RMSE: 39024.41461633466
MAE: 15322.641341772493
R2 score: 27.332369989970708

Model Testing Performance
RMSE: 39724.55462628288
MAE: 15484.581225382037
R2 score: 26.4713105799279


Decision Tree
Model Training Performance
RMSE: 1536.8080665064163
MAE: 89.44410824666839
R2 score: 99.88730426904392

Model Testing Performance
RMSE: 20547.936832541924
MAE: 7283.807315691939
R2 score: 80.32679146408157


Random Forest Regressor
Model Training Performance
RMSE: 6495.515448381233
MAE: 2141.1765279038855
R2 score: 97.98676050356649

M

Hyper Parameter tuning

In [13]:
final_models={

    'Random Forest Regressor':RandomForestRegressor(),
    'GradientBoostingRegressor':GradientBoostingRegressor(),
    'XGBRegressor':XGBRegressor(),
    'LGBMRegressor':LGBMRegressor(),
    'CatBoosting Regressor':CatBoostRegressor(verbose=False),
}

In [14]:
param_grids = {
    'RandomForestRegressor': {
        'n_estimators': [100, 300, 500, 700],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'max_depth': [3, 5, 7, 10],
        'min_samples_leaf': [1, 2, 4],
        'min_samples_split': [2, 5, 10],
        'max_features': ['sqrt', 'log2', None],
    },

    'CatBoostRegressor': {
        'depth': [4, 6, 8],
        'iterations': [200, 500],
        'learning_rate': [0.01, 0.1, 0.2],
        'l2_leaf_reg': [3, 5, 6, 7, 8]
    },

    'XGBRegressor': {
        'n_estimators': [100, 200, 500],
        'learning_rate': [0.01, 0.1, 0.2, 0.05],
        'max_depth': [3, 5, 7],
        'colsample_bytree': [0.7, 0.5],
        'reg_alpha': [0.5, 1],
        'reg_lambda': [1.0, 0.5],
        'min_child_weight': [1, 2, 3]
    },

    'GradientBoostingRegressor': {
        'n_estimators': [100, 200, 500]
    },

    'LGBMRegressor': {
        'n_estimators': [100, 200, 500],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [5, 10, 15],
        'num_leaves': [31, 20, 15],
        'force_col_wise': [True, False],
        'reg_alpha': [0.5, 1],
        'reg_lambda': [0.5, 1],
        'min_child_samples': [5,10,15,20]
    }
}

In [15]:
trained_model_list = []
model_list = []
r2_list = []

for model_name, model in final_models.items():
    print(f"Tuning {model_name}...")

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grids.get(model_name, {}),
        n_iter=5,
        scoring='r2',
        cv=3,
        n_jobs=-1,
        random_state=42
    )

    # Fit with hyperparameter tuning
    search.fit(X_train_encoded, y_train)

    # Get best model
    best_model = search.best_estimator_
    trained_model_list.append(best_model)
    model_list.append(model_name)

    # Evaluate on train set
    y_train_pred = best_model.predict(X_train_encoded)
    mae_train, rmse_train, r2_train = evaluate_model(y_train, y_train_pred)

    # Evaluate on test set
    y_test_pred = best_model.predict(X_test_encoded)
    mae_test, rmse_test, r2_test = evaluate_model(y_test, y_test_pred)

    r2_list.append(r2_test)

    print(f"Best Params for {model_name}: {search.best_params_}")
    print(f"Train -> MAE: {mae_train:.4f}, RMSE: {rmse_train:.4f}, R2: {r2_train*100:.2f}%")
    print(f"Test  -> MAE: {mae_test:.4f}, RMSE: {rmse_test:.4f}, R2: {r2_test*100:.2f}%\n")


Tuning Random Forest Regressor...


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=5. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best Params for Random Forest Regressor: {}
Train -> MAE: 2131.9850, RMSE: 6467.2908, R2: 98.00%
Test  -> MAE: 5670.0983, RMSE: 16272.9694, R2: 87.66%

Tuning GradientBoostingRegressor...


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 3 is smaller than n_iter=5. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best Params for GradientBoostingRegressor: {'n_estimators': 500}
Train -> MAE: 6424.4644, RMSE: 14777.1410, R2: 89.58%
Test  -> MAE: 6831.3437, RMSE: 17249.3366, R2: 86.14%

Tuning XGBRegressor...
Best Params for XGBRegressor: {'reg_lambda': 1.0, 'reg_alpha': 0.5, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.5}
Train -> MAE: 4816.6957, RMSE: 10346.9154, R2: 94.89%
Test  -> MAE: 5764.8746, RMSE: 15686.1290, R2: 88.54%

Tuning LGBMRegressor...
[LightGBM] [Info] Total Bins 953
[LightGBM] [Info] Number of data points in the train set: 47185, number of used features: 11
[LightGBM] [Info] Start training from score 35313.027403
Best Params for LGBMRegressor: {'reg_lambda': 0.5, 'reg_alpha': 1, 'num_leaves': 15, 'n_estimators': 500, 'min_child_samples': 20, 'max_depth': 15, 'learning_rate': 0.2, 'force_col_wise': True}
Train -> MAE: 4932.8651, RMSE: 10933.4948, R2: 94.30%
Test  -> MAE: 5843.7663, RMSE: 15906.7045, R2: 88.21%

Tuning C

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=5. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best Params for CatBoosting Regressor: {}
Train -> MAE: 5381.4094, RMSE: 10767.3395, R2: 94.47%
Test  -> MAE: 6095.7662, RMSE: 16036.3645, R2: 88.02%

